In [1]:
%cd ..

/Users/n-zagainov/kolobok/ml


In [ ]:
from pathlib import Path

import pandas as pd

[07/05/25 19:02:45] WARNING  Your inference package version 0.51.0 is out of date! Please upgrade to ]8;id=186917;file:///Users/n-zagainov/.local/share/mamba/envs/klbk/lib/python3.9/site-packages/inference/core/__init__.py\__init__.py]8;;\:]8;id=701332;file:///Users/n-zagainov/.local/share/mamba/envs/klbk/lib/python3.9/site-packages/inference/core/__init__.py#41\41]8;;\
                             version 0.51.2 of inference for the latest features and bug fixes by                  
                             running `pip install --upgrade inference`.                                            

/Users/n-zagainov/.local/share/mamba/envs/klbk/lib/python3.9/site-packages/inference/models/utils.py:341: ModelDependencyMissing: Your `inference` configuration does not support Qwen2.5-VL model. Use pip install 'inference[transformers]' to install missing requirements.
  warnings.warn(
/Users/n-zagainov/.local/share/mamba/envs/klbk/lib/python3.9/site-packages/inference/models/utils.py:353: ModelDependencyMissing: Your `inference` configuration does not support SAM model. Use pip install 'inference[sam]' to install missing requirements.
  warnings.warn(
/Users/n-zagainov/.local/share/mamba/envs/klbk/lib/python3.9/site-packages/inference/models/utils.py:363: ModelDependencyMissing: Your `inference` configuration does not support SAM model. Use pip install 'inference[sam]' to install missing requirements.
  warnings.warn(
/Users/n-zagainov/.local/share/mamba/envs/klbk/lib/python3.9/site-packages/inference/models/utils.py:374: ModelDependencyMissing: Your `inference` configuration does no

In [42]:
data_path = Path("data")

input_path = data_path / "annotations"
output_path = data_path / "annotations_processed"
output_path.mkdir(parents=True, exist_ok=True)

index_path = data_path / "models/models.csv"

df = pd.read_csv(index_path, sep=";")
df_brand = df[df["parent_id"] == 0][["id", "name"]].rename(columns={"id": "brand_id", "name": "brand_name"})
df = df[df["parent_id"] != 0].merge(df_brand, left_on="parent_id", right_on="brand_id", how="left")[["id", "name", "brand_id", "brand_name"]]

In [116]:
### query column by a string
from difflib import SequenceMatcher
from functools import partial
import json


def save_result(img_number, model_id, brand_id, force=False):
    res = {
        "model": model_id,
        "brand": brand_id,
    }
    save_path = output_path / f"{img_number}.json"
    if save_path.exists() and not force:
        print("SAVE PATH EXISTS! CANCEL SAVING")
        return
    with open(save_path, "w") as f:
        json.dump(res, f)

    print(f"saved to {save_path}")


def get_ratio(a, b):
    return SequenceMatcher(None, str(a).lower(), str(b).lower()).ratio()

def display_comparison(model_name=None, brand_name=None):
    if model_name == "":
        model_name = None
    if brand_name == "":
        brand_name = None
    if model_name is None and brand_name is None:
        raise Exception("At least one of model_name or brand_name must be provided")
    
    result_df = df.copy()
    
    if model_name is not None:
        # Calculate similarity ratio for model names
        get_ratio_partial = partial(get_ratio, b=model_name)
        result_df["model_ratio"] = result_df["name"].apply(get_ratio_partial)
    
    if brand_name is not None:
        # Calculate similarity ratio for brand names
        get_ratio_partial = partial(get_ratio, b=brand_name)
        result_df["brand_ratio"] = result_df["brand_name"].apply(get_ratio_partial)
    
    # Determine which ratio to use for sorting
    if model_name is not None and brand_name is not None:
        # If both are provided, use the average of both ratios
        result_df["ratio"] = (result_df["model_ratio"] + result_df["brand_ratio"]) / 2
    elif model_name is not None:
        result_df["ratio"] = result_df["model_ratio"]
    else:
        result_df["ratio"] = result_df["brand_ratio"]
    
    # Clean up intermediate columns
    if "model_ratio" in result_df.columns:
        result_df = result_df.drop("model_ratio", axis=1)
    if "brand_ratio" in result_df.columns:
        result_df = result_df.drop("brand_ratio", axis=1)
    
    return result_df.sort_values("ratio", ascending=False).head(15)


In [143]:
display_comparison(
    model_name="mr-hp172",
    brand_name="pirelli",
)

,id,name,brand_id,brand_name,ratio
5893,6986,MR-HP172,267,Mirage,0.730769
2223,3316,P7,32,Pirelli,0.700000
5894,6987,MR-HT172,267,Mirage,0.668269
2224,3317,P700,32,Pirelli,0.666667
2226,3319,P7000,32,Pirelli,0.653846
2191,3284,Cinturato P7,32,Pirelli,0.650000
2187,3280,Cinturato P1,32,Pirelli,0.650000
2225,3318,P700-Z,32,Pirelli,0.642857
2186,3279,Cinturato CN12,32,Pirelli,0.636364
5896,6989,MR-W172,267,Mirage,0.630769


In [144]:
save_result(
    img_number=19,
    model_id=6986,
    brand_id=267,
)

saved to data/annotations_processed/19.json
